# Colab 재현 노트북: 논문-특허 통합 방법론 구현

이 노트북은 아래 논문의 **절차와 주요 하이퍼파라미터를 최대한 그대로** 따라가되, 당신의 데이터 스키마와 분석 구간(**1975–2025**)에 맞게 조정한 Colab용 버전입니다.

- 논문 데이터: `final_paper_data_with_CR.csv`
- 특허 데이터: `filtered_patent_data_ready.csv`
- 분석 범위: **1975–2025 (inclusive)**
- 분석 방식: **논문/특허를 먼저 각각 분석한 뒤, 마지막에 비교·갭 분석**
- 핵심 하이퍼파라미터: `alpha=0.01`, `beta=0.01`, `minimum_probability=0.001`, `chunksize=100`, `iterations=10000`

> 주의: 원논문은 2002–2021 구간에서 5개 시계열 구간을 사용했습니다. 이 노트북은 같은 4년 단위 구조를 유지하기 위해 1975–2025를 **13개 기간(T01–T13)** 으로 나눕니다. 즉, **방법론/하이퍼파라미터는 최대한 동일하게 유지**하고, **기간 정의만 사용자 데이터에 맞게 확장**했습니다.

## 현재 데이터 기준 기본 현황

- 논문 레코드(1975–2025): **10,008건**
- 특허 레코드(1975–2025): **1,865건**

### 기간별 문서 수
| Period | Years | Papers | Patents |
|---|---:|---:|---:|
| T01 | 1975-1978 | 54 | 8 |
| T02 | 1979-1982 | 88 | 8 |
| T03 | 1983-1986 | 74 | 20 |
| T04 | 1987-1990 | 111 | 31 |
| T05 | 1991-1994 | 239 | 48 |
| T06 | 1995-1998 | 357 | 40 |
| T07 | 1999-2002 | 490 | 102 |
| T08 | 2003-2006 | 538 | 146 |
| T09 | 2007-2010 | 825 | 153 |
| T10 | 2011-2014 | 1,074 | 143 |
| T11 | 2015-2018 | 1,430 | 183 |
| T12 | 2019-2022 | 2,278 | 581 |
| T13 | 2023-2025 | 2,450 | 402 |

## 중요: Colab 오류 수정 반영

이 수정본은 `numpy.dtype size changed` 오류를 피하기 위해 **numpy / pandas를 직접 고정 설치하지 않도록** 바꾼 버전입니다.
첫 설치 셀을 실행하면 런타임이 자동 재시작되며, 그 다음 맨 위부터 다시 실행하면 됩니다.

In [ ]:
# Colab-safe 패키지 설치
# 핵심: numpy/pandas를 강제로 건드리지 않아서 ABI 충돌을 피합니다.
%pip -q install --upgrade pip setuptools wheel
%pip -q install --upgrade gensim stanza rapidfuzz pyLDAvis pyyaml tqdm

import os
print('설치 완료. ABI 충돌 방지를 위해 런타임을 자동 재시작합니다.')
os.kill(os.getpid(), 9)


In [1]:
# NLTK / Stanza 리소스 다운로드
import nltk, stanza
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
stanza.download('en', processors='tokenize,pos,lemma,depparse', verbose=False)
print('환경 준비 완료')

환경 준비 완료


## 데이터 불러오기

아래 두 방식 중 하나를 쓰면 됩니다.

1. **Colab에 직접 업로드**
2. **Google Drive 마운트 후 경로 지정**

기본 코드는 업로드 방식을 사용합니다.

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import json
import yaml
from dataclasses import dataclass
from collections import Counter
from tqdm import tqdm

PAPER_PATH = Path('/content/final_paper_data_with_CR.csv')
PATENT_PATH = Path('/content/filtered_patent_data_ready.csv')
OUTPUT_DIR = Path('./outputs_cultured_meat_1975_2025')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 논문 구조 + 사용자 범위에 맞춘 기간 정의
PERIODS = [{'name': 'T01', 'start_year': 1975, 'end_year': 1978, 'topic_range': [1, 10]},
    {'name': 'T02', 'start_year': 1979, 'end_year': 1982, 'topic_range': [1, 10]},
    {'name': 'T03', 'start_year': 1983, 'end_year': 1986, 'topic_range': [10, 30]},
    {'name': 'T04', 'start_year': 1987, 'end_year': 1990, 'topic_range': [10, 30]},
    {'name': 'T05', 'start_year': 1991, 'end_year': 1994, 'topic_range': [10, 30]},
    {'name': 'T06', 'start_year': 1995, 'end_year': 1998, 'topic_range': [10, 30]},
    {'name': 'T07', 'start_year': 1999, 'end_year': 2002, 'topic_range': [10, 30]},
    {'name': 'T08', 'start_year': 2003, 'end_year': 2006, 'topic_range': [10, 30]},
    {'name': 'T09', 'start_year': 2007, 'end_year': 2010, 'topic_range': [10, 30]},
    {'name': 'T10', 'start_year': 2011, 'end_year': 2014, 'topic_range': [10, 30]},
    {'name': 'T11', 'start_year': 2015, 'end_year': 2018, 'topic_range': [10, 30]},
    {'name': 'T12', 'start_year': 2019, 'end_year': 2022, 'topic_range': [10, 30]},
    {'name': 'T13', 'start_year': 2023, 'end_year': 2025, 'topic_range': [10, 30]}
]

CONFIG = {
    'project_name': 'cultured_meat_paper_patent_replication',
    'random_seed': 42,
    'input': {
        'papers_path': str(PAPER_PATH),
        'patents_path': str(PATENT_PATH),
        'text_columns': {
            'title': 'title',
            'abstract': 'abstract',
            'year': 'year',
            'id': 'id'
        }
    },
    'output': {'base_dir': str(OUTPUT_DIR)},
    'periods': PERIODS,
    'preprocessing': {
        'lowercase': True,
        'merge_title_abstract': True,
        'remove_publisher_patterns': ['All rights reserved', 'Elsevier', 'Springer', 'Copyright'],
        'custom_stopwords': ['this', 'paper', 'study', 'article', 'patent', 'invention', 'method', 'result', 'based', 'using', 'used'],
        'min_token_len': 2,
        'keep_pos_prefixes': ['N', 'J', 'V']
    },
    'lda': {
        'alpha': 0.01,
        'beta': 0.01,
        'minimum_probability': 0.001,
        'chunksize': 100,
        'iterations': 10000,
        'passes': 1,
        'eval_every': 0,
        'topn_terms': 10,
        'no_below': 2,
        'no_above': 0.95,

         'min_docs_per_period': 5,
          'sparse_doc_threshold': 20,
          'sparse_no_below': 1,
          'sparse_no_above': 1.0
    },
    'sao': {
        'parser_backend': 'stanza',
        'stanza_lang': 'en',
        'expand_terms_with_lemmas': True,
        'fuzzy_merge_threshold': 90,
        'max_sentences_per_doc': 40,
        'min_sao_frequency': 1
    },
    'subsystems': {
        # 원논문의 전고체전지용 키워드는 배양육 데이터에 그대로 맞지 않으므로,
        # 여기서는 데이터셋에 맞춰 수정 가능한 placeholder로 둡니다.
        'system_1_keywords': ['cell', 'cellular', 'stem', 'myoblast', 'muscle', 'fat', 'adipocyte', 'fibroblast'],
        'system_2_keywords': ['scaffold', 'hydrogel', 'bioreactor', 'culture', 'medium', 'serum', 'growth factor'],
        'system_3_keywords': ['texture', 'structure', '3d', 'printing', 'coating', 'alignment', 'differentiation']
    },
    'trend': {
        'latest_period': 'T13',
        'cross_source_match_threshold': 0.2,
        'top_terms_per_topic_for_matching': 10
    }
}

with open(OUTPUT_DIR / 'config_used.yaml', 'w', encoding='utf-8') as f:
    yaml.safe_dump(CONFIG, f, allow_unicode=True, sort_keys=False)

print('CONFIG 저장 완료:', OUTPUT_DIR / 'config_used.yaml')

CONFIG 저장 완료: outputs_cultured_meat_1975_2025/config_used.yaml


## 사용자 데이터 스키마를 논문 코드 스키마로 정규화

- 논문 CSV: `UT (Unique WOS ID)`, `Article Title`, `Abstract`, `Publication Year`
- 특허 CSV: `출원번호`, `출원일`, `발명의 명칭-번역문`, `요약-번역문`

특허는 영어 NLP 파이프라인을 맞추기 위해 **번역문 우선**으로 사용하고, 비어 있으면 원문 컬럼으로 fallback 하게 구성했습니다.

In [3]:
# 데이터 로드 및 스키마 정규화
papers_raw = pd.read_csv(PAPER_PATH)
patents_raw = pd.read_csv(PATENT_PATH)

papers = pd.DataFrame({
    'id': papers_raw['UT (Unique WOS ID)'].astype(str),
    'title': papers_raw['Article Title'].fillna('').astype(str),
    'abstract': papers_raw['Abstract'].fillna('').astype(str),
    'year': pd.to_numeric(papers_raw['Publication Year'], errors='coerce')
})

patent_title = patents_raw['발명의 명칭-번역문'].fillna('').astype(str)
patent_abs = patents_raw['요약-번역문'].fillna('').astype(str)

if '발명의 명칭' in patents_raw.columns:
    patent_title = patent_title.mask(patent_title.str.strip().eq(''), patents_raw['발명의 명칭'].fillna('').astype(str))
if '요약' in patents_raw.columns:
    patent_abs = patent_abs.mask(patent_abs.str.strip().eq(''), patents_raw['요약'].fillna('').astype(str))

patents = pd.DataFrame({
    'id': patents_raw['출원번호'].astype(str),
    'title': patent_title,
    'abstract': patent_abs,
    'year': pd.to_datetime(patents_raw['출원일'], errors='coerce').dt.year
})

papers = papers[papers['year'].between(1975, 2025)].copy()
patents = patents[patents['year'].between(1975, 2025)].copy()

papers['year'] = papers['year'].astype(int)
patents['year'] = patents['year'].astype(int)

print('papers shape:', papers.shape)
print('patents shape:', patents.shape)
print('papers year range:', papers['year'].min(), papers['year'].max())
print('patents year range:', patents['year'].min(), patents['year'].max())

papers.head(2), patents.head(2)

papers shape: (10008, 4)
patents shape: (1865, 4)
papers year range: 1975 2025
patents year range: 1975 2025


(                    id                                              title  \
 0  WOS:001427449100001  Reassessing the sustainability promise of cult...   
 1  WOS:001422651100001  Review: Livestock cell types with myogenic dif...   
 
                                             abstract  year  
 0  There are currently over 170 companies in the ...  2025  
 1  With the current environmental impact of large...  2025  ,
             id                                title  \
 0  2024-718203  비-동물성 단백질 식품 제품을 위한 결합제로서의 글루코아밀라아제   
 1  2017-817928                     육류 처리 조성물과 이의 사용   
 
                                             abstract  year  
 0  본 발명은 식품 분야에 관한 것이다. 본 발명은 상기 식품의 제조에 사용된 성분에 ...  2024  
 1  본 발명은 부분적으로 또는 완전히 중화된 아세트산의 형태로 완충된 식품 산 성분의 ...  2017  )

In [4]:
# 정규화된 입력 저장 (파이프라인 재사용용)
papers.to_csv(OUTPUT_DIR / 'papers_normalized.csv', index=False)
patents.to_csv(OUTPUT_DIR / 'patents_normalized.csv', index=False)
print('정규화 CSV 저장 완료')

정규화 CSV 저장 완료


In [5]:
# ===== helper functions =====
import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from gensim import corpora
from gensim.models import LdaModel
from rapidfuzz import fuzz

LEMMATIZER = WordNetLemmatizer()

def normalize_text(text: str) -> str:
    text = '' if pd.isna(text) else str(text)
    text = text.replace('\n', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def clean_text(text: str, publisher_patterns=None):
    text = normalize_text(text).lower()
    publisher_patterns = publisher_patterns or []
    for p in publisher_patterns:
        text = re.sub(re.escape(p.lower()), ' ', text)
    text = re.sub(r'[^a-z0-9\-\./\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def get_stopwords(custom_stopwords=None):
    sw = set(stopwords.words('english'))
    sw.update(custom_stopwords or [])
    return sw

def tokenize_and_filter(text, stop_words, min_token_len=2, keep_pos_prefixes=None):
    keep_pos_prefixes = tuple(keep_pos_prefixes or ['N', 'J', 'V'])
    tokens = nltk.word_tokenize(text)
    tagged = nltk.pos_tag(tokens)
    out = []
    for token, pos in tagged:
        token = token.lower().strip()
        if len(token) < min_token_len:
            continue
        if token in stop_words:
            continue
        if not re.match(r'^[a-z0-9\-\.]+$', token):
            continue
        if not pos.startswith(keep_pos_prefixes):
            continue
        lemma = LEMMATIZER.lemmatize(token)
        if lemma in stop_words or len(lemma) < min_token_len:
            continue
        out.append(lemma)
    return out

def sentence_split(text: str):
    return nltk.sent_tokenize(normalize_text(text))

def expand_term(term: str):
    term = normalize_text(term).lower()
    variants = {term}
    parts = term.split()
    if parts:
        lemma_parts = [LEMMATIZER.lemmatize(p) for p in parts]
        variants.add(' '.join(lemma_parts))
    if term.endswith('ies'):
        variants.add(term[:-3] + 'y')
    if term.endswith('s') and len(term) > 3:
        variants.add(term[:-1])
    return {v.strip() for v in variants if v.strip()}

def assign_period(year, periods):
    for p in periods:
        if int(p['start_year']) <= int(year) <= int(p['end_year']):
            return p['name']
    return None

In [6]:
@dataclass
class CorpusBundle:
    raw: pd.DataFrame
    processed: pd.DataFrame

def preprocess_dataframe(df, config, source_name):
    cols = config['input']['text_columns']
    pp = config['preprocessing']
    stop_words = get_stopwords(pp.get('custom_stopwords', []))

    work = df.copy()
    work[cols['title']] = work[cols['title']].fillna('')
    work[cols['abstract']] = work[cols['abstract']].fillna('')
    work[cols['year']] = pd.to_numeric(work[cols['year']], errors='coerce')
    work = work[work[cols['year']].notna()].copy()
    work[cols['year']] = work[cols['year']].astype(int)
    work['source'] = source_name
    work['period'] = work[cols['year']].apply(lambda y: assign_period(y, config['periods']))
    work = work[work['period'].notna()].copy()
    work['text_raw'] = (work[cols['title']] + '. ' + work[cols['abstract']]).str.strip()
    work['text_clean'] = work['text_raw'].apply(lambda x: clean_text(x, pp.get('remove_publisher_patterns', [])))
    work['tokens'] = work['text_clean'].apply(
        lambda x: tokenize_and_filter(
            x,
            stop_words=stop_words,
            min_token_len=pp.get('min_token_len', 2),
            keep_pos_prefixes=pp.get('keep_pos_prefixes', ['N', 'J', 'V'])
        )
    )
    work['token_count'] = work['tokens'].apply(len)
    work = work[work['token_count'] > 0].copy()
    return CorpusBundle(raw=df, processed=work)

In [7]:
from pathlib import Path
import pandas as pd
import json
from gensim import corpora
from gensim.models import LdaModel

class LDAPeriodRunner:
    def __init__(self, config):
        self.config = config
        self.lda_cfg = config['lda']

    def _dictionary_corpus(self, docs_tokens):
        # 빈 토큰 문서 제거
        docs_tokens = [tokens for tokens in docs_tokens if isinstance(tokens, list) and len(tokens) > 0]
        n_docs = len(docs_tokens)

        dictionary = corpora.Dictionary(docs_tokens)

        # 희소한 초기 구간은 필터를 완화
        if n_docs < self.lda_cfg.get('sparse_doc_threshold', 20):
            no_below = self.lda_cfg.get('sparse_no_below', 1)
            no_above = self.lda_cfg.get('sparse_no_above', 1.0)
        else:
            no_below = self.lda_cfg.get('no_below', 2)
            no_above = self.lda_cfg.get('no_above', 0.95)

        dictionary.filter_extremes(no_below=no_below, no_above=no_above)

        corpus = [dictionary.doc2bow(tokens) for tokens in docs_tokens]
        corpus = [bow for bow in corpus if len(bow) > 0]

        return dictionary, corpus

    def _fit_lda(self, corpus, dictionary, num_topics):
        if len(dictionary) == 0 or len(corpus) == 0:
            raise ValueError("Empty dictionary/corpus after filtering")

        # 토픽 수는 문서 수 / 단어 수를 넘지 않도록 안전장치
        safe_num_topics = max(1, min(num_topics, len(corpus), len(dictionary)))

        return LdaModel(
            corpus=corpus,
            id2word=dictionary,
            num_topics=safe_num_topics,
            alpha=self.lda_cfg['alpha'],
            eta=self.lda_cfg['beta'],
            minimum_probability=self.lda_cfg['minimum_probability'],
            chunksize=min(self.lda_cfg.get('chunksize', 100), max(1, len(corpus))),
            iterations=self.lda_cfg['iterations'],
            passes=self.lda_cfg.get('passes', 1),
            eval_every=self.lda_cfg.get('eval_every', 0),
            random_state=self.config.get('random_seed', 42),
        )

    def search_best_topic_count(self, df_period, topic_min, topic_max):
        docs_tokens = df_period['tokens'].tolist()
        dictionary, corpus = self._dictionary_corpus(docs_tokens)

        # 사전/코퍼스가 비면 skip
        if len(dictionary) == 0 or len(corpus) == 0:
            return None, pd.DataFrame(), dictionary, corpus

        # 토픽 수를 데이터 크기에 맞게 안전하게 제한
        max_possible_topics = max(1, min(topic_max, len(corpus), len(dictionary)))
        min_possible_topics = max(1, min(topic_min, max_possible_topics))

        results = []
        for k in range(min_possible_topics, max_possible_topics + 1):
            try:
                lda = self._fit_lda(corpus, dictionary, k)
                perplexity = lda.log_perplexity(corpus)
                results.append({
                    'num_topics': k,
                    'log_perplexity': perplexity
                })
            except Exception as e:
                results.append({
                    'num_topics': k,
                    'log_perplexity': None,
                    'error': str(e)
                })

        perf_df = pd.DataFrame(results)

        valid_perf_df = perf_df[perf_df['log_perplexity'].notna()].copy()
        if valid_perf_df.empty:
            return None, perf_df, dictionary, corpus

        best_row = valid_perf_df.sort_values('log_perplexity', ascending=True).iloc[0]
        return int(best_row['num_topics']), perf_df, dictionary, corpus

    def extract_top_terms(self, lda_model, topn):
        out = []
        for topic_id in range(lda_model.num_topics):
            terms = lda_model.show_topic(topic_id, topn=topn)
            out.append({
                'topic_id': topic_id,
                'terms': [t for t, _ in terms],
                'weights': [float(w) for _, w in terms]
            })
        return out

    def infer_doc_topics(self, lda_model, dictionary, docs_tokens):
        rows = []
        valid_doc_index = 0

        for idx, tokens in enumerate(docs_tokens):
            if not isinstance(tokens, list) or len(tokens) == 0:
                continue

            bow = dictionary.doc2bow(tokens)
            if len(bow) == 0:
                continue

            dist = lda_model.get_document_topics(bow, minimum_probability=0.0)
            for topic_id, prob in dist:
                rows.append({
                    'doc_index': valid_doc_index,
                    'original_doc_index': idx,
                    'topic_id': topic_id,
                    'topic_prob': float(prob)
                })
            valid_doc_index += 1

        return pd.DataFrame(rows)

    def run_source(self, df, source_name, output_dir):
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

        summary_rows = []
        min_docs_per_period = self.lda_cfg.get('min_docs_per_period', 5)

        for p in self.config['periods']:
            p_name = p['name']
            df_p = df[df['period'] == p_name].copy()

            if df_p.empty:
                continue

            print(f'[{source_name}] {p_name} docs =', len(df_p))

            # 문서가 너무 적으면 스킵
            if len(df_p) < min_docs_per_period:
                print(f'[{source_name}] {p_name} skipped: too few documents (< {min_docs_per_period})')
                continue

            k_best, perf_df, dictionary, corpus = self.search_best_topic_count(
                df_p,
                p['topic_range'][0],
                p['topic_range'][1]
            )

            # dictionary/corpus가 비면 스킵
            if k_best is None:
                print(f'[{source_name}] {p_name} skipped: empty dictionary/corpus after filtering')
                if not perf_df.empty:
                    perf_df.to_csv(output_dir / f'{source_name}_{p_name}_perplexity.csv', index=False)
                continue

            lda = self._fit_lda(corpus, dictionary, k_best)
            top_terms = self.extract_top_terms(lda, self.lda_cfg.get('topn_terms', 10))
            doc_topics = self.infer_doc_topics(lda, dictionary, df_p['tokens'].tolist())

            if not perf_df.empty:
                perf_df.to_csv(output_dir / f'{source_name}_{p_name}_perplexity.csv', index=False)

            with open(output_dir / f'{source_name}_{p_name}_top_terms.json', 'w', encoding='utf-8') as f:
                json.dump(top_terms, f, ensure_ascii=False, indent=2)

            doc_topics.to_csv(output_dir / f'{source_name}_{p_name}_doc_topics.csv', index=False)
            dictionary.save(str(output_dir / f'{source_name}_{p_name}.dict'))
            lda.save(str(output_dir / f'{source_name}_{p_name}.lda'))

            for topic in top_terms:
                summary_rows.append({
                    'source': source_name,
                    'period': p_name,
                    'optimal_topics': k_best,
                    'topic_id': topic['topic_id'],
                    'top_terms': '; '.join(topic['terms'])
                })

        summary_df = pd.DataFrame(summary_rows)
        summary_df.to_csv(output_dir / f'{source_name}_topic_summary.csv', index=False)
        return summary_df


In [8]:
class SAOExtractor:
    def __init__(self, config):
        self.config = config
        self.sao_cfg = config['sao']
        self.parser = None

    def load(self):
        self.parser = stanza.Pipeline(
            lang=self.sao_cfg.get('stanza_lang', 'en'),
            processors='tokenize,pos,lemma,depparse',
            tokenize_no_ssplit=False,
            verbose=False
        )
        return self

    def _extract_stanza_triples(self, sentence):
        doc = self.parser(sentence)
        triples = []
        for sent in doc.sentences:
            words = sent.words
            by_head = {}
            for w in words:
                by_head.setdefault(w.head, []).append(w)
            for w in words:
                if w.upos in {'VERB', 'AUX'}:
                    subs = [x for x in by_head.get(w.id, []) if x.deprel in {'nsubj', 'nsubj:pass'}]
                    objs = [x for x in by_head.get(w.id, []) if x.deprel in {'obj', 'iobj', 'obl'}]
                    for s in subs:
                        for o in objs:
                            triples.append((s.text.lower(), w.lemma.lower(), o.text.lower()))
        return triples

    def build_term_set(self, term_list):
        term_set = set()
        for term in term_list:
            term_set.add(term.lower())
            if self.sao_cfg.get('expand_terms_with_lemmas', True):
                term_set.update(expand_term(term))
        return term_set

    def fuzzy_merge(self, sao_df):
        threshold = self.sao_cfg.get('fuzzy_merge_threshold', 90)
        unique_texts = list(dict.fromkeys(sao_df['sao_text'].tolist()))
        canonical = {}
        for txt in unique_texts:
            assigned = None
            for c in canonical:
                if fuzz.token_sort_ratio(txt, c) >= threshold:
                    assigned = c
                    break
            canonical[txt] = assigned or txt
        sao_df = sao_df.copy()
        sao_df['sao_merged'] = sao_df['sao_text'].map(canonical)
        return sao_df

    def extract_from_df(self, df, term_df, output_dir, source_name):
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        all_rows = []

        for period in sorted(df['period'].unique()):
            df_p = df[df['period'] == period].copy()
            term_rows = term_df[(term_df['source'] == source_name) & (term_df['period'] == period)]
            terms = []
            for _, row in term_rows.iterrows():
                terms.extend([x.strip() for x in row['top_terms'].split(';') if x.strip()])
            term_set = self.build_term_set(terms)

            for _, row in tqdm(df_p.iterrows(), total=len(df_p), desc=f'{source_name}-{period}-sao'):
                sentences = sentence_split(row['text_raw'])[: self.sao_cfg.get('max_sentences_per_doc', 40)]
                for sent in sentences:
                    sent_l = sent.lower()
                    if not any(t in sent_l for t in term_set):
                        continue
                    triples = self._extract_stanza_triples(sent)
                    for s, a, o in triples:
                        joined = f'{s} | {a} | {o}'
                        if any(t in s or t in o or t in sent_l for t in term_set):
                            all_rows.append({
                                'source': source_name,
                                'period': period,
                                'doc_id': row.get('id', None),
                                'sentence': sent,
                                'subject': s,
                                'action': a,
                                'object': o,
                                'sao_text': joined
                            })

        sao_df = pd.DataFrame(all_rows).drop_duplicates()
        if sao_df.empty:
            sao_df.to_csv(output_dir / f'{source_name}_sao_raw.csv', index=False)
            return sao_df

        sao_df = self.fuzzy_merge(sao_df)
        sao_df.to_csv(output_dir / f'{source_name}_sao_merged.csv', index=False)
        freq = sao_df.groupby(['period', 'sao_merged']).size().reset_index(name='freq')
        freq.to_csv(output_dir / f'{source_name}_sao_frequency.csv', index=False)
        return sao_df

def sao_to_tokens(sao_text):
    return [normalize_text(x) for x in sao_text.replace('|', ' ').split() if x.strip()]

In [9]:
def compute_topic_intensity(doc_topics_df):
    return doc_topics_df.groupby('topic_id')['topic_prob'].mean().reset_index(name='intensity')

def compute_tip(topic_intensity_df):
    ati = topic_intensity_df['intensity'].mean()
    out = topic_intensity_df.copy()
    out['ati'] = ati
    out['tip'] = out['intensity'] / ati if ati else 0.0
    return out

def compute_gri(doc_topic_assignments_by_period, latest_period_name, periods_order):
    latest_idx = periods_order.index(latest_period_name)
    window = periods_order[max(0, latest_idx - 4): latest_idx + 1]
    rows = []
    all_df = []
    for period_name, df in doc_topic_assignments_by_period.items():
        temp = df.copy()
        temp['period'] = period_name
        all_df.append(temp)
    all_df = pd.concat(all_df, ignore_index=True)
    counts = all_df.groupby(['period', 'topic_id'])['doc_index'].nunique().reset_index(name='doc_count')

    latest_counts = counts[counts['period'] == latest_period_name]
    for _, row in latest_counts.iterrows():
        topic_id = row['topic_id']
        numerator = row['doc_count']
        denom = counts[(counts['topic_id'] == topic_id) & (counts['period'].isin(window))]['doc_count'].sum()
        gri = numerator / denom if denom else 0.0
        rows.append({'topic_id': topic_id, 'gri': gri, 'latest_doc_count': numerator, 'window_doc_count': denom})
    return pd.DataFrame(rows)

def parse_top_terms(s):
    return set([x.strip().lower() for x in str(s).split(';') if x.strip()])

def jaccard(a, b):
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

def compare_cross_source_topics(paper_summary, patent_summary, latest_period='T13', threshold=0.2):
    p1 = paper_summary[paper_summary['period'] == latest_period].copy()
    p2 = patent_summary[patent_summary['period'] == latest_period].copy()
    matches = []
    for _, r1 in p1.iterrows():
        t1 = parse_top_terms(r1['top_terms'])
        best = None
        best_score = -1
        for _, r2 in p2.iterrows():
            t2 = parse_top_terms(r2['top_terms'])
            score = jaccard(t1, t2)
            if score > best_score:
                best_score = score
                best = r2
        matches.append({
            'paper_topic_id': r1['topic_id'],
            'paper_top_terms': r1['top_terms'],
            'patent_topic_id': None if best is None else best['topic_id'],
            'patent_top_terms': None if best is None else best['top_terms'],
            'match_score': best_score,
            'is_potential_gap_or_opportunity': best_score < threshold
        })
    return pd.DataFrame(matches)

def quadrant_label(tip, gri):
    high_tip = tip >= 1.0
    high_gri = gri >= 0.5
    if high_tip and high_gri:
        return 'I_high_intensity_high_growth'
    if (not high_tip) and high_gri:
        return 'II_low_intensity_high_growth'
    if high_tip and (not high_gri):
        return 'III_high_intensity_low_growth'
    return 'IV_low_intensity_low_growth'

def assemble_trend_table(tip_df, gri_df):
    out = tip_df.merge(gri_df, on='topic_id', how='left').fillna({'gri': 0.0})
    out['quadrant'] = out.apply(lambda r: quadrant_label(r['tip'], r['gri']), axis=1)
    return out.sort_values(['quadrant', 'tip', 'gri'], ascending=[True, False, False])

## 1단계. 전처리 실행

In [10]:
papers_bundle = preprocess_dataframe(papers, CONFIG, 'papers')
patents_bundle = preprocess_dataframe(patents, CONFIG, 'patents')

papers_bundle.processed.to_csv(OUTPUT_DIR / 'papers_preprocessed.csv', index=False)
patents_bundle.processed.to_csv(OUTPUT_DIR / 'patents_preprocessed.csv', index=False)

print('papers processed:', papers_bundle.processed.shape)
print('patents processed:', patents_bundle.processed.shape)
papers_bundle.processed[['id','year','period','token_count']].head()

papers processed: (10008, 10)
patents processed: (706, 10)


,id,year,period,token_count
0,WOS:001427449100001,2025,T13,87
1,WOS:001422651100001,2025,T13,151
2,WOS:001426217900001,2025,T13,146
3,WOS:001422846800001,2025,T13,132
4,WOS:001524785300001,2025,T13,170


## 2단계. 기간별 LDA 실행

원논문과 동일하게:
- 첫 두 기간: topic 수 탐색 범위 `1–10`
- 이후 기간: topic 수 탐색 범위 `10–30`
- 선택 기준: `log_perplexity` 최소

In [11]:
lda_runner = LDAPeriodRunner(CONFIG)
paper_summary = lda_runner.run_source(papers_bundle.processed, 'papers', OUTPUT_DIR / 'lda')
patent_summary = lda_runner.run_source(patents_bundle.processed, 'patents', OUTPUT_DIR / 'lda')

print('paper topics:', paper_summary.shape)
print('patent topics:', patent_summary.shape)
paper_summary.head()

[papers] T01 docs = 54


[papers] T02 docs = 88


[papers] T03 docs = 74


[papers] T04 docs = 111


[papers] T05 docs = 239


...

[patents] T11 docs = 77


[patents] T12 docs = 183


[patents] T13 docs = 149


paper topics: (129, 5)
patent topics: (232, 5)


,source,period,optimal_topics,topic_id,top_terms
0,papers,T01,4,0,ms.; female; production; wk; culture; exposure...
1,papers,T01,4,1,meat; strain; c.; culture; sausage; type; samp...
2,papers,T01,4,2,growth; meat; c.; culture; system; medium; wei...
3,papers,T01,4,3,culture; meat; salmonella; medium; virus; stra...
4,papers,T02,4,0,meat; culture; medium; bacteria; growth; acid;...


## 3단계. SAO 추출

주의: 이 단계가 가장 오래 걸립니다. 데이터가 크기 때문에 처음에는 최근 1~2개 기간만 돌려 테스트한 뒤 전체 실행하는 것을 권장합니다.

In [12]:
sao_extractor = SAOExtractor(CONFIG).load()
papers_sao = sao_extractor.extract_from_df(papers_bundle.processed, paper_summary, OUTPUT_DIR / 'sao', 'papers')
patents_sao = sao_extractor.extract_from_df(patents_bundle.processed, patent_summary, OUTPUT_DIR / 'sao', 'patents')

print('papers_sao:', papers_sao.shape)
print('patents_sao:', patents_sao.shape)
papers_sao.head()

patents-T13-sao: 100%|██████████| 149/149 [00:22<00:00,  6.66it/s]


papers_sao: (87636, 9)
patents_sao: (0, 0)


,source,period,doc_id,sentence,subject,action,object,sao_text,sao_merged
0,papers,T01,WOS:A1978FU24500009,"The ability of the meat starter cultures, Pedi...",ability,examine,range,ability | examine | range,ability | examine | range
1,papers,T01,WOS:A1978FU24500009,"The ability of the meat starter cultures, Pedi...",ability,examine,degree,ability | examine | degree,ability | examine | degree
2,papers,T01,WOS:A1978FL87600016,The nitrite reacted with the meat myoglobin to...,nitrite,react,myoglobin,nitrite | react | myoglobin,nitrite | react | myoglobin
3,papers,T01,WOS:A1977EC23900021,A new liquid medium (modified Lombard-Dowell b...,medium,inoculate,strains,medium | inoculate | strains,medium | inoculate | strains
4,papers,T01,WOS:A1977EC23900021,Both broths were subcultured at 48 and 72 h to...,broths,subculture,48,broths | subculture | 48,broths | subculture | 48


## 4단계. SAO 토큰 기반 재클러스터링

In [ ]:
def rerun_lda_on_sao(config, sao_df, source_name, base_dir):
    if sao_df.empty:
        return pd.DataFrame(), {}
    work = sao_df.copy()
    work['tokens'] = work['sao_merged'].apply(sao_to_tokens)
    runner = LDAPeriodRunner(config)
    summary = runner.run_source(work, f'{source_name}_sao', base_dir / 'sao_lda')

    period_doc_topics = {}
    for p in config['periods']:
        p_name = p['name']
        f = base_dir / 'sao_lda' / f'{source_name}_sao_{p_name}_doc_topics.csv'
        if f.exists():
            period_doc_topics[p_name] = pd.read_csv(f)
    return summary, period_doc_topics

paper_sao_summary, paper_doc_topics = rerun_lda_on_sao(CONFIG, papers_sao, 'papers', OUTPUT_DIR)
patent_sao_summary, patent_doc_topics = rerun_lda_on_sao(CONFIG, patents_sao, 'patents', OUTPUT_DIR)

print('paper_sao_summary:', paper_sao_summary.shape)
print('patent_sao_summary:', patent_sao_summary.shape)

## 5단계. 트렌드 분석 (TIP / GRI / 논문-특허 갭)

In [ ]:
latest_period = CONFIG['trend']['latest_period']
periods_order = [p['name'] for p in CONFIG['periods']]

trend_outputs = {}
for source_name, summary, period_doc_topics in [
    ('papers', paper_sao_summary, paper_doc_topics),
    ('patents', patent_sao_summary, patent_doc_topics),
]:
    if latest_period not in period_doc_topics:
        continue
    latest_df = period_doc_topics[latest_period]
    intensity = compute_topic_intensity(latest_df)
    tip_df = compute_tip(intensity)
    gri_df = compute_gri(period_doc_topics, latest_period, periods_order)
    trend_df = assemble_trend_table(tip_df, gri_df)
    trend_df.to_csv(OUTPUT_DIR / 'trend' / f'{source_name}_trend_metrics.csv', index=False)
    trend_outputs[source_name] = trend_df

compare_df = compare_cross_source_topics(
    paper_sao_summary,
    patent_sao_summary,
    latest_period=latest_period,
    threshold=CONFIG['trend']['cross_source_match_threshold']
)
(OUTPUT_DIR / 'trend').mkdir(parents=True, exist_ok=True)
compare_df.to_csv(OUTPUT_DIR / 'trend' / 'cross_source_topic_gaps.csv', index=False)

print('trend outputs saved')
compare_df.head()

## 6단계. 빠른 시각화

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

if 'papers' in trend_outputs:
    plt.figure(figsize=(8,6))
    df = trend_outputs['papers'].copy()
    sns.scatterplot(data=df, x='tip', y='gri', hue='quadrant', s=100)
    plt.title('Paper Topics: TIP vs GRI')
    plt.show()

if 'patents' in trend_outputs:
    plt.figure(figsize=(8,6))
    df = trend_outputs['patents'].copy()
    sns.scatterplot(data=df, x='tip', y='gri', hue='quadrant', s=100)
    plt.title('Patent Topics: TIP vs GRI')
    plt.show()

## 7단계. 산출물 압축

In [ ]:
import shutil
archive_path = shutil.make_archive('cultured_meat_replication_outputs', 'zip', OUTPUT_DIR)
print('압축 완료:', archive_path)

## 중요 메모

이 노트북은 **논문의 절차와 핵심 파라미터를 최대한 맞춘 1차 재현 버전**입니다. 다만 아래 두 부분은 논문과 완전 동일하지 않습니다.

1. **Stanford Parser 대신 Stanza dependency parser 사용**
   - Colab에서 실제 구동성과 재현성을 위해 대체했습니다.
2. **기술 서브시스템 분류**
   - 원논문은 전고체전지 도메인 전문가 검토를 포함합니다.
   - 당신의 데이터(배양육 관련)에서는 키워드 사전이나 수작업 라벨링 규칙을 추가로 다듬는 것이 좋습니다.

즉, **LDA → 기술용어 추출 → SAO → SAO 재클러스터링 → 시계열 비교 → 갭 분석**의 큰 틀은 그대로 유지하되, 도메인 지식이 필요한 부분은 배양육 분야에 맞게 2차 조정하는 것이 가장 좋습니다.